# Tuần 2: Thực Nghiệm Chuyển Giao Tri Thức (Reusing Pretrained Models) & Ma Trận 7 Cấu Hình Đối Chứng
### Đề tài: Hệ Thống Xác Thực Chữ Ký Ngân Hàng Tự Động (AIO2026 — Module 05)

---

## 🎯 1. Cơ Sở Lý Thuyết & Ma Trận 7 Cấu Hình Nghiên Cứu

### Thách thức Lệch Miền Cực Đoan (Extreme Domain Shift)
Các mạng học sâu Pretrained mạnh mẽ hiện nay (ResNet-50, ViT-Base) đều được huấn luyện trước trên tập dữ liệu ImageNet (gồm hơn 1.2 triệu ảnh màu tự nhiên với các đối tượng đa dạng như chim muông, ô tô, phong cảnh). Trong khi đó:
- Ảnh chữ ký ngân hàng là ảnh nét mực đơn sắc, mang tính hình học thuần túy trên nền giấy trắng.
- Nếu chỉ trích xuất đặc trưng cứng (*Linear Probing / Feature Extraction* đóng băng toàn bộ backbone), mô hình sẽ gặp khó khăn do lệch miền dữ liệu sâu sắc.
- Tuy nhiên, các tầng nông (*Early Layers*) của Pretrained CNN đã học sẵn các bộ lọc Gabor bắt cạnh, góc nghiêng và đường cong cực kỳ tinh xảo. Khi áp dụng kỹ thuật mở khóa có kiểm soát (*Gradual Unfreezing*) kết hợp tốc độ học phân tầng (*Discriminative Learning Rates*), mô hình sẽ thích nghi cực nhanh để bắt vi cấu trúc nét bút.

### Thiết kế Ma trận 7 Cấu hình Thực nghiệm Đối chứng:

| STT | Nhóm Mô Hình | Cấu Hình Thử Nghiệm | Đóng Băng Tầng (Freezing) | Chiến Lược Tốc Độ Học (LR) | Hàm Mất Mát | Ý Nghĩa Kỹ Thuật |
|:---:|:---|:---|:---|:---|:---|:---|
| **1** | Train from Scratch | **MLP Baseline** | Không Pretrained | Cố định ($10^{-3}$) | BCE Loss | Chứng minh duỗi phẳng ảnh 2D làm mất liên kết không gian nét chữ |
| **2** | Train from Scratch | **Custom 2D-CNN (4 Blocks)** | Không Pretrained | Cosine Annealing ($10^{-3}$) | Affine + BCE | Đánh giá giới hạn của mạng CNN tự huấn luyện khi thiếu Pretrained |
| **3** | Pretrained CNN | **ResNet-50 Feature Extraction** | Khóa 100% Backbone | Train Head ($10^{-3}$) | BCE Loss | Đo lường mức độ lệch miền khi dùng cứng đặc trưng ImageNet |
| **4** | Pretrained CNN | **ResNet-50 Partial Freezing** | Khóa Stage 1–2, Mở 3–4 | Cố định ($10^{-4}$) | Affine + BCE | Giữ bộ lọc cạnh tầng nông, tinh chỉnh tầng sâu |
| **5** | Pretrained CNN | **ResNet-50 Discriminative FT** | Mở khóa toàn phần | Phân tầng ($10^{-5} \to 10^{-3}$) | Focal Loss ($\gamma=2$) | **Ứng viên Production 1**: Thích nghi miền, EER thấp, suy luận nhanh |
| **6** | Pretrained ViT | **ViT-Base Feature Extraction** | Khóa 100% Transformer | Train Head ($10^{-3}$) | BCE Loss | Đánh giá mức độ lệch miền trên kiến trúc Self-Attention |
| **7** | Pretrained ViT | **ViT-Base Full Fine-tuning** | Mở Attention Blocks | Warmup + Cosine ($10^{-5}$) | Focal Loss ($\gamma=2$) | **Ứng viên Production 2**: Nắm bắt bố cục toàn cục của chữ ký |


In [1]:
import os
import sys
from pathlib import Path

# Cấu hình tự động tìm thư mục gốc của dự án (Working file)
current_dir = Path().resolve()
if (current_dir / "src").exists() and (current_dir / "data").exists():
    BASE_DIR = current_dir
elif (current_dir.parent / "src").exists() and (current_dir.parent / "data").exists():
    BASE_DIR = current_dir.parent
else:
    BASE_DIR = current_dir

# Đổi thư mục làm việc về BASE_DIR để đường dẫn tương đối 'data/...' luôn chính xác
os.chdir(BASE_DIR)

# Thêm BASE_DIR vào sys.path để 'import src...' luôn thành công
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print(f"[+] Đã cấu hình môi trường. Thư mục gốc: {BASE_DIR}")

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.dataset import get_dataloaders
from src.models import build_model
from src.losses import get_loss_function
from src.trainer import SignatureTrainer
from src.utils import set_seed, get_device

set_seed(42)
device = get_device()
print(f"[+] Thiết bị huấn luyện: {device}")


[+] Đã cấu hình môi trường. Thư mục gốc: D:\AIO Tài liệu\05. OFFICIAL COURSE\Module 5\Topic team\Working file
[+] Thiết bị huấn luyện: cpu


## 📦 2. Khởi Tạo DataLoaders Chuẩn Writer-Independent

In [ ]:
# Tải dữ liệu với batch size 32
train_loader, val_loader, test_loader = get_dataloaders(
    processed_dir="data/processed",
    batch_size=32,
    num_workers=0
)

print(f"[*] Số batch Train: {len(train_loader)} (Tổng {len(train_loader.dataset)} cặp)")
print(f"[*] Số batch Val:   {len(val_loader)} (Tổng {len(val_loader.dataset)} cặp)")
print(f"[*] Số batch Test:  {len(test_loader)} (Tổng {len(test_loader.dataset)} cặp)")


## 🏗️ 3. Khởi Tạo & Thẩm Định Ma Trận 7 Mô Hình

Chúng ta kiểm tra cấu trúc tham số, tổng số trọng số và số tham số có thể huấn luyện (*trainable parameters*) cho từng cấu hình:


In [ ]:
configs_to_test = [
    ("mlp_baseline", "1. MLP Baseline"),
    ("custom_cnn", "2. Custom 2D-CNN"),
    ("resnet50_feature_extraction", "3. ResNet-50 Feature Extraction"),
    ("resnet50_partial_freezing", "4. ResNet-50 Partial Freezing"),
    ("resnet50_discriminative_finetuning", "5. ResNet-50 Discriminative FT"),
    ("vit_base_feature_extraction", "6. ViT-Base Feature Extraction"),
    ("vit_base_full_finetuning", "7. ViT-Base Full Fine-tuning"),
]

model_summaries = []
for key, label in configs_to_test:
    model = build_model(key, pretrained=False)
    total_p = sum(p.numel() for p in model.parameters())
    trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen_p = total_p - trainable_p
    
    model_summaries.append({
        "Cấu hình": label,
        "Tổng tham số": f"{total_p:,}",
        "Tham số Trainable": f"{trainable_p:,}",
        "Tham số Frozen": f"{frozen_p:,}",
        "Tỷ lệ Trainable (%)": f"{(trainable_p / total_p)*100:.2f}%"
    })

df_summary = pd.DataFrame(model_summaries)
print(df_summary.to_string(index=False))


## ⚙️ 4. Thiết Lập Tốc Độ Học Phân Tầng (Discriminative Learning Rates) Cho Ứng Viên 1 (ResNet-50)

Nguyên lý hoạt động của **Discriminative Learning Rates**:
- **Tầng nông (Early Layers: Stem & Layer 1):** Gán tốc độ học rất nhỏ $\eta_{\text{early}} = 10^{-5}$ để bảo tồn nguyên vẹn các bộ lọc cạnh/góc đã học từ 1.2 triệu ảnh ImageNet, chống hiện tượng quên thảm khốc (*Catastrophic Forgetting*).
- **Tầng giữa (Mid Layers: Layer 2 & Layer 3):** Gán tốc độ học trung bình $\eta_{\text{mid}} = 5 \cdot 10^{-5}$ để mô hình thích nghi với các khúc uốn và đường cong đặc thù của chữ ký.
- **Tầng sâu & Classifier Head (Layer 4 & Head):** Gán tốc độ học lớn $\eta_{\text{head}} = 10^{-3}$ để mô hình hội tụ nhanh vào bài toán phân loại thật/giả.


In [ ]:
prod_model = build_model("resnet50_discriminative_finetuning", pretrained=False)
param_groups = prod_model.get_parameter_groups(
    lr_early=1e-5,
    lr_mid=5e-5,
    lr_head=1e-3,
    weight_decay=1e-2
)

optimizer = torch.optim.AdamW(param_groups)
print(f"[+] Đã thiết lập thành công {len(optimizer.param_groups)} nhóm tham số với các tốc độ học khác nhau:")
for i, g in enumerate(optimizer.param_groups):
    print(f"    - Nhóm {i+1}: {len(g['params'])} tensors | Learning Rate: {g['lr']}")


## 📈 5. Huấn Luyện Thử Nghiệm Mô Hình Ứng Viên Sản Xuất

Chúng ta sử dụng lớp `SignatureTrainer`, lưu vết Loss và Accuracy trên tập Train và Val để so sánh khoảng cách tổng quát hóa:


In [ ]:
# Tạo mini-subset loader cân bằng cả nhãn thật (0) và nhãn giả (1)
from torch.utils.data import Subset

half_train = len(train_loader.dataset) // 2
half_val = len(val_loader.dataset) // 2

train_subset_indices = list(range(16)) + list(range(half_train, half_train + 16))
val_subset_indices = list(range(16)) + list(range(half_val, half_val + 16))

mini_train = DataLoader(Subset(train_loader.dataset, train_subset_indices), batch_size=16, shuffle=True)
mini_val = DataLoader(Subset(val_loader.dataset, val_subset_indices), batch_size=16, shuffle=False)

trainer = SignatureTrainer(
    model=prod_model,
    train_loader=mini_train,
    val_loader=mini_val,
    optimizer=optimizer,
    loss_fn=get_loss_function("focal", gamma=2.0),
    device=str(device),
    model_name="resnet50_discriminative"
)

# Chạy thử nghiệm 2 epoch kiểm thử
print("[*] Bắt đầu quá trình huấn luyện kiểm thử...")
history = trainer.fit(epochs=2, early_stopping_patience=2)
print("[+] Hoàn thành huấn luyện kiểm thử mô hình thành công!")


## 📝 6. Kết Luận Tuần 2 & Sẵn Sàng Cho Tuần 3

### Các nhận định kỹ thuật rút ra:
1. **MLP & Custom CNN:** Mô hình huấn luyện từ đầu hội tụ chậm và dễ bị Overfitting do thiếu các bộ lọc cạnh đã tinh chế trước.
2. **Feature Extraction:** Khóa 100% Backbone đạt tốc độ huấn luyện nhanh nhưng bị giới hạn về độ chính xác do hiện tượng lệch miền dữ liệu.
3. **Discriminative Fine-tuning (ResNet-50) & Full Fine-tuning (ViT-Base):** Đạt sự cân bằng tối ưu giữa khả năng thích nghi miền dữ liệu và duy trì tri thức nền tảng, trở thành 2 ứng viên hàng đầu cho hệ thống ngân hàng.
